PROMPT ENGINEERING IN LANGSMITH

Import environment variables

In [1]:
import os
from dotenv import load_dotenv

_ = load_dotenv(dotenv_path=".env", override=True)
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")

Pull in Prompt from Prompthub

In [ ]:
from langsmith import Client

client = Client(api_key=LANGSMITH_API_KEY)
prompt = client.pull_prompt("essay_5yo_consice", include_model=True, secrets={"OPENAI_API_KEY": os.getenv("OPENAI_API_KEY")})

d:\repos\langgraph_learn\venv\Lib\site-packages\langchain_core\load\load.py:822: UserWarning: WARNING! extra_headers is not default parameter.
                extra_headers was transferred to model_kwargs.
                Please confirm that extra_headers is what you intended.
  loaded_obj = {k: _load(v) for k, v in obj.items()}


Setup AI Application

In [3]:
# Init web search tool
from tavily import TavilyClient

tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

Let's now crate our application, same as in the tracing module. This time, our prompt is the one pulled from `PromptHub`

In [4]:
from langchain_openai import ChatOpenAI
from langsmith import traceable

llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))

@traceable
def search(question):
  web_response = tavily.search(query=question, max_results=3)
  return "\n".join([d["content"] for d in web_response["results"]])

@traceable
def explain(question, context):
  result = prompt.invoke({"question": question, "context": context}) # very easy to use this prompt and paste params which are intialized inside the prompt
  return result.content

@traceable
def main(question):
  context = search(question)
  answer = explain(question, context)
  return answer


Test Application

In [5]:
question = "what is complexity economics?"
print(main(question))

KeyError: "Input to ChatPromptTemplate is missing variables {'answer'}.  Expected: ['answer', 'question'] Received: ['question', 'context']\nNote: if you intended {answer} to be part of the string and not a variable, please escape it with double curly braces like: '{{answer}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "